Lab 3: Logistic Regression as a Soft Decision Model

In [2]:
import pandas as pd
import numpy as np

# TODO: Load the glass dataset
df = pd.read_csv("glass.csv")

# TODO: Check number of rows and columns
print(f"Dataset Shape: {df.shape}")

# TODO: See column names and first few rows
print(df.columns)
print(df.head())

Dataset Shape: (214, 10)
Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')
        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1


Which column is the output we want to predict? The Type column is the original output.

Are all columns numeric? Yes, the glass dataset consists of refractive index and chemical compositions (Na, Mg, Al, Si, K, Ca, Ba, Fe) which are all numeric.

Is there an ID column that should not be used? Typically, the first column in this dataset is an ID. If present, it must be dropped as it contains no predictive chemical information.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TODO: Create binary labels (Type 1 is positive class)
df["y"] = (df["Type"] == 1).astype(int)

# TODO: Remove original Type column and ID column if it exists
df = df.drop(columns=["Type"])
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

# TODO: Separate features and labels
X = df.drop(columns=["y"]).values
y = df["y"].values

# TODO: Split data into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
# 9.1 Sigmoid Function
def sigmoid(z):
    # Converts score z into a probability between 0 and 1
    return 1 / (1 + np.exp(-z))

# 9.2 Forward Computation
def predict_proba(X, w, b):
    # Compute z (evidence score) and convert to probability p
    z = X @ w + b
    p = sigmoid(z)
    return p

# 10. Loss Function (Binary Cross Entropy)
def loss(y, p):
    # Penalizes the "wrongness" of the confidence
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# 11. Learning Step (Weight Update)
def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y
    # Update weights and bias based on gradient
    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)
    return w, b

In [6]:
# 12. Training Loop
w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1
epochs = 100

for i in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)
    if i % 10 == 0:
        current_p = predict_proba(X_train, w, b)
        print(f"Epoch {i}, Loss: {loss(y_train, current_p)}")

Epoch 0, Loss: 0.6821572833929108
Epoch 10, Loss: 0.6107403583486661
Epoch 20, Loss: 0.5747883497537751
Epoch 30, Loss: 0.5528994016173656
Epoch 40, Loss: 0.5379208733110972
Epoch 50, Loss: 0.5269060291929022
Epoch 60, Loss: 0.5184063241889877
Epoch 70, Loss: 0.5116157378700156
Epoch 80, Loss: 0.5060453170923557
Epoch 90, Loss: 0.5013794317202315


In [7]:
# 13. From Probability to Decision
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

# Testing with different thresholds
p_test = predict_proba(X_test, w, b)
y_pred_05 = predict_label(p_test, 0.5)
y_pred_07 = predict_label(p_test, 0.7)

Why a higher threshold (0.7) is safer in glass quality control:
A higher threshold means the model must be more confident (70% belief) before it labels a glass as "Type 1" (Accept). This is safer because it reduces "False Positives"—situations where we might accept a low-quality or incorrect glass type just because the model was 51% sure.

Perceptron vs. Logistic Regression

This model differs from the Perceptron because it outputs a continuous probability rather than a hard binary label. The Sigmoid function matters because it preserves information near the decision boundary, allowing the model to express uncertainty rather than forcing a jump from 0 to 1. However, a remaining problem is that while we have calibrated the output, the model is still inherently linear; it may still struggle with complex, non-linear patterns in the chemical data.